# LangGraph：暂停后到底从哪里继续？

使用真实 LangGraph 1.2.15，CPU 即可，无模型 API。约 15 分钟。先猜计数，再运行验证。

本示例以 InMemorySaver 保留内存状态，**没有跨进程持久化能力**；生产恢复需要持久 checkpointer。无真实发布或写入操作。

In [ ]:
import os, sys, subprocess
# 仅安装依赖；不连接模型、不申请 Drive 权限。
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'langgraph==1.2.15'])
os.environ['LANGSMITH_TRACING'] = 'false'
os.environ['LANGCHAIN_TRACING_V2'] = 'false'


## 1. 在 interrupt 前后各计数一次

第一次 invoke 应只执行 interrupt 前的代码。恢复会重新进入节点；计数器用来暴露重入现象，不是生产实现中的可靠计数。

In [ ]:
from typing import TypedDict
from collections import Counter
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

class State(TypedDict):
    draft: str
    approved: bool

calls = Counter()
def approve_draft(state: State):
    calls['before_interrupt'] += 1
    approved = interrupt({'draft': state['draft'], 'question': '是否继续？'})
    calls['after_interrupt'] += 1
    return {'approved': bool(approved)}

def build_graph(saver):
    builder = StateGraph(State)
    builder.add_node('approve', approve_draft)
    builder.add_edge(START, 'approve')
    builder.add_edge('approve', END)
    return builder.compile(checkpointer=saver)

saver = InMemorySaver()
graph = build_graph(saver)
config = {'configurable': {'thread_id': 'study-001'}}
paused = graph.invoke({'draft': '比较三篇论文的方法', 'approved': False}, config)
assert paused['__interrupt__'][0].value['draft'] == '比较三篇论文的方法'
assert graph.get_state(config).next == ('approve',)
assert dict(calls) == {'before_interrupt': 1}
print('暂停：', dict(calls), '待恢复节点：', graph.get_state(config).next)

finished = graph.invoke(Command(resume=True), config)
assert finished['approved'] is True
assert dict(calls) == {'before_interrupt': 2, 'after_interrupt': 1}
assert graph.get_state(config).next == ()
print('恢复：', dict(calls), 'approved =', finished['approved'])


## 2. 验证拒绝、线程和存储边界

同一个 thread_id 才能找回同一检查点。新内存 saver 不含旧状态，不能拿这个实验声称实现了进程崩溃恢复。

In [ ]:
# 不同 thread_id 不会继承上一条会话。
other = {'configurable': {'thread_id': 'study-002'}}
assert graph.get_state(other).values == {}
graph.invoke({'draft': '另一个草稿', 'approved': False}, other)
rejected = graph.invoke(Command(resume=False), other)
assert rejected['approved'] is False

# 同一内存 saver 被新图复用可以读到状态；新 saver 则读不到。
assert build_graph(saver).get_state(config).values['approved'] is True
assert build_graph(InMemorySaver()).get_state(config).values == {}
print('边界检查通过：拒绝分支、线程隔离、新 saver 没有旧状态。')

import json
from importlib.metadata import version
report = {'langgraph': version('langgraph'), 'calls': dict(calls),
          'same_thread_approved': finished['approved'], 'other_thread_approved': rejected['approved'],
          'new_saver_state': build_graph(InMemorySaver()).get_state(config).values}
assert report['langgraph'] == '1.2.15'
with open('langgraph_report.json', 'w') as file:
    json.dump(report, file, ensure_ascii=False, indent=2)
print(json.dumps(report, ensure_ascii=False, indent=2))


## 3. 修改一次再解释

将 resume=True 改成 False，检查 approved。把第一次与恢复的 thread_id 改成不同值，观察错误或空状态；不要把它当作正确恢复方案。

思考：如果 interrupt 前是付费 API 调用，恢复后会发生什么？应把外部操作放在审批后，并使用稳定操作 ID 与查询/对账协议。

Files 中可下载 langgraph_report.json，自己的解释可以贴回网站练习。

[官方 Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts) · [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence)

本例已在本机隔离环境逐单元执行；Colab 首次运行需安装包，托管运行环境可能变化。